# Query CouchDB pakai Mango

Di notebook ini semua operasi esensial (R1–R10, W1–W10, plus PF-01, PF-03, dan PF-07 yang tidak ada di
Part 2) ditulis ulang pakai Mango, bahasa query bawaan CouchDB. Datanya data penuh 2 juta order, dan setiap
hasil dicek pakai validasi yang sama dengan benchmark.

Mango itu query berbentuk JSON yang dikirim ke `POST /{database}/_find`, mirip
`SELECT ... WHERE ... ORDER BY ... LIMIT` di SQL. Tapi ada beberapa batasan. Mango cuma bisa baca data, jadi
untuk insert, update, dan delete tetap pakai `PUT`, `_bulk_docs`, atau `DELETE`. Mango juga tidak bisa
`GROUP BY`, `COUNT`, atau `SUM`. Selain itu, kalau tidak ada index, CouchDB harus cek dokumen satu per satu,
dan untuk 2 juta order itu makan waktu sekitar 130 detik.

Karena itu tidak semua operasi bisa full Mango:
- R10 dan PF-01-Q01 bisa full Mango.
- W1–W10 dan PF-07 pakai Mango untuk cari dokumennya, lalu disimpan pakai `PUT`, `_bulk_docs`, atau `DELETE`.
- PF-01-Q02, PF-03-Q01, dan R1–R9 tetap pakai view. Query Mango-nya tetap ditulis, tapi yang dijalankan cuma
  `_explain`-nya untuk lihat kenapa tidak bisa.

Semua query Mango ditulis sebagai JSON biasa, jadi bisa langsung di-copy ke Fauxton
(`http://127.0.0.1:5984/_utils`, menu Run A Query with Mango).

Sebelum dijalankan, pastikan container CouchDB sudah nyala, dataset canonical sudah di-load, dan file
`data/reference/couchdb_reference_canonical.json` sudah ada. Operasi tulis pakai data uji benchmark
(ID `...9999...`) yang langsung dihapus lagi setelah selesai, jadi data canonical tidak berubah.

Contekan SQL ke Mango:

| SQL | Mango |
|---|---|
| `FROM service_orders` | database `grab_orders`, ditambah `"type": "service_order"` |
| `WHERE a = 'x' AND b = 'y'` | `"selector": {"a": "x", "b": "y"}` |
| `WHERE a <> 'x'`, `a IN (...)` | `{"a": {"$ne": "x"}}`, `{"a": {"$in": [...]}}` |
| `WHERE a >= 1 AND a < 9` | `{"a": {"$gte": 1, "$lt": 9}}` |
| `JOIN` ke tabel anak (trips, payment) | field tertanam: `"payment.payment_status"`, `"trips": {"$elemMatch": {...}}` |
| `SELECT a, b` | `"fields": ["a", "b"]` |
| `ORDER BY a` | `"sort": [{"a": "asc"}]` (wajib ada index) |
| `LIMIT 20` | `"limit": 20` |
| `EXPLAIN` | `POST /{db}/_explain` dengan query yang sama |
| `INSERT`, `UPDATE`, `DELETE` | `PUT`, `_bulk_docs`, `DELETE` (bukan Mango) |
| `GROUP BY`, `COUNT`, `SUM` | view map/reduce (bukan Mango) |

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('COUCHDB_REFERENCE_FILE',
                      str(ROOT / 'data' / 'reference' / 'couchdb_reference_canonical.json'))

from benchmarks.adapters import couchdb as adapter
from couchdb import model
from couchdb.client import CouchError
from couchdb.queries import OPERATIONS, GrabCouch

env = adapter.env()                  # klien admin (data uji) dan grab_app (semua query)
app, admin = env.app, env.admin
ORDERS, FLEET, CUSTOMERS, GEO = (env.settings.db(n) for n in ('orders', 'fleet', 'customers', 'geo'))
grab = GrabCouch(app, env.settings)  # implementasi di couchdb/queries.py, dipakai untuk bagian view
SCENARIOS = {s.operation_id: s for s in adapter.create_scenarios()}
RESULTS = {}

pd.set_option('display.max_colwidth', 50)
print(f"CouchDB {admin.server_info()['version']}, query dijalankan sebagai {app.user}")

CouchDB 3.5.3, query dijalankan sebagai grab_app


Beberapa fungsi bantu supaya cell di bawah lebih pendek. `mango()` menjalankan query lalu menampilkan index
yang dipakai, waktunya, dan berapa dokumen yang dicek. `explain()` cuma menampilkan rencana eksekusinya tanpa
menjalankan query. `put()` menyimpan satu dokumen, dan kalau `_rev`-nya sudah tidak sama dengan yang di
database, CouchDB menolak dengan error 409. `start()` dan `finish()` menyiapkan data uji, memvalidasi hasil
dengan aturan benchmark, lalu menghapus data ujinya lagi.

In [2]:
def mango(db, query):
    print(f'POST /{db}/_find')
    started = time.perf_counter()
    result = app.find(db, {**query, 'execution_stats': True})
    elapsed = (time.perf_counter() - started) * 1000
    index = app.json('POST', f'/{db}/_explain', body=query)['index']
    used = 'tanpa index Mango (_all_docs)' if index['type'] == 'special' else f"{index['ddoc']} / {index['name']}"
    examined = result.get('execution_stats', {}).get('total_docs_examined', 0)
    print(f"-> {len(result['docs'])} dokumen dalam {elapsed:,.1f} ms | index: {used} | "
          f"dokumen diperiksa: {examined:,}")
    return result['docs']


def explain(db, query):
    print(f'POST /{db}/_explain')
    index = app.json('POST', f'/{db}/_explain', body=query).get('index')
    if not index:
        print('-> tidak ada index yang bisa dipakai')
    elif index['type'] == 'special':
        print('-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database')
    else:
        print(f"-> index: {index['ddoc']} / {index['name']}")


def put(db, doc):
    old = doc.get('_rev', '(dokumen baru)')
    new = app.put_doc(db, doc)
    print(f"PUT /{db}/{doc['_id']}: _rev {old} -> {new}")


def timed(call, label):
    started = time.perf_counter()
    result = call()
    print(f'{label}: {(time.perf_counter() - started) * 1000:,.1f} ms')
    return result


def find_trip(order, trip_id):
    return next(t for t in order['trips'] if t['trip_id'] == trip_id)


def find_assignment(order, assignment_id):
    for trip in order['trips']:
        for assignment in trip['assignments']:
            if assignment['assignment_id'] == assignment_id:
                return trip, assignment


def new_trip(trip_id, start, end):
    return {'trip_id': trip_id, 'trip_status': 'SEARCHING', 'started_at': None, 'completed_at': None,
            'start_location_id': start['_id'], 'end_location_id': end['_id'],
            'start_service_area_id': start['service_area_id'], 'assignments': []}


def start(operation_id):
    scenario = SCENARIOS[operation_id]
    return scenario, scenario.setup()


def finish(scenario, ctx, result):
    try:
        passed = scenario.validate(ctx, result)
    finally:
        scenario.teardown(ctx)
    RESULTS[scenario.operation_id] = passed
    cleaned = '' if scenario.scenario in ('read', 'analytics') else ' Data uji sudah dihapus.'
    print(f'\n{scenario.operation_id}: lolos validasi = {passed}.{cleaned}')

## 1. Baca data operasional (PF-01, PF-03, PF-05)

### PF-01-Q01: baca satu pesanan lengkap

Ini tidak ada di Part 2. Kalau di SQL, untuk dapat satu pesanan lengkap harus join `service_orders` dengan
subtipe, `payments`, `trips`, dan `driver_assignments`. Di CouchDB semuanya sudah ada di satu dokumen, jadi
cukup dicari berdasarkan `_id`.

In [3]:
sc, ctx = start('PF-01-Q01')
query = json.loads('''
{
  "selector": {"_id": "ORD0000001"}
}
''')
assert query['selector']['_id'] == ctx['params']['order_id']
order = mango(ORDERS, query)[0]
print(f"{order['_id']}: {order['service_type']} {order['order_status']}, payment "
      f"{order['payment']['payment_status']}, {len(order['trips'])} trip")
finish(sc, ctx, order)

POST /grab_orders/_find
-> 1 dokumen dalam 153.6 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
ORD0000001: MOBILITY CANCELLED, payment REFUNDED, 1 trip

PF-01-Q01: lolos validasi = True.


### PF-01-Q02: 20 pesanan terbaru seorang customer

Query Mango-nya gampang ditulis, tapi ternyata ditolak CouchDB. `sort` di Mango harus ada index-nya,
sedangkan kita tidak punya index untuk `customer_id` + `created_at`. Kalau `sort`-nya dibuang pun, query ini
tetap harus cek semua 2 juta order. Makanya implementasinya pakai view `ops/by_customer`, yang key-nya
`[customer_id, created_at]`.

In [4]:
sc, ctx = start('PF-01-Q02')
query = json.loads('''
{
  "selector": {"type": "service_order", "customer_id": "CUS0195816"},
  "fields": ["_id", "created_at", "order_status", "service_type", "total_amount"],
  "sort": [{"created_at": "desc"}],
  "limit": 20
}
''')
assert query['selector']['customer_id'] == ctx['params']['customer_id']
try:
    app.find(ORDERS, query)
except CouchError as error:
    print('Mango ditolak CouchDB:', error)
explain(ORDERS, {k: v for k, v in query.items() if k != 'sort'})
result = timed(lambda: grab.customer_order_history(**ctx['params']), 'view ops/by_customer')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

Mango ditolak CouchDB: POST /grab_orders/_find -> 400 no_usable_index: No index exists for this sort, try indexing by the sort fields.
POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view ops/by_customer: 13.0 ms


,order_id,created_at,order_status,service_type,total_amount
0,ORD1999744,2026-10-31T23:59:37Z,CONFIRMED,MOBILITY,49000
1,ORD1995948,2026-10-31T23:53:59Z,CONFIRMED,DELIVERY,57000
2,ORD1995278,2026-10-31T23:53:00Z,CONFIRMED,FOOD,57500
3,ORD1993213,2026-10-31T23:49:56Z,IN_PROGRESS,DELIVERY,36000
4,ORD1991230,2026-10-31T23:46:59Z,CONFIRMED,FOOD,52000



PF-01-Q02: lolos validasi = True.


### PF-03-Q01: status perjalanan dari `trip_id`

Trip disimpan di dalam dokumen order, jadi di Mango dicari pakai `$elemMatch` (cocok kalau ada satu elemen
array yang memenuhi syarat). Masalahnya sama seperti sebelumnya: tidak ada index, jadi harus scan semua
order. Implementasinya pakai view `ops/by_trip`.

In [5]:
sc, ctx = start('PF-03-Q01')
query = json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {"trip_id": "TRP0000001"}}
  },
  "fields": ["_id", "order_status", "trips"]
}
''')
assert query['selector']['trips']['$elemMatch']['trip_id'] == ctx['params']['trip_id']
explain(ORDERS, query)
result = timed(lambda: grab.trip_status(**ctx['params']), 'view ops/by_trip')
print(json.dumps(result, indent=2))
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view ops/by_trip: 4.3 ms
{
  "trip_id": "TRP0000001",
  "order_id": "ORD0000001",
  "trip_status": "CANCELLED",
  "started_at": null,
  "completed_at": null,
  "order_status": "CANCELLED"
}

PF-03-Q01: lolos validasi = True.


### PF-05-Q01 (R10): outlet yang OPEN di Tebet, Jakarta

SQL Part 2 (R10):
```sql
SELECT
    mo.merchant_outlet_id,
    mo.outlet_name,
    l.address,
    l.latitude,
    l.longitude
FROM service_areas sa
JOIN locations l ON l.service_area_id = sa.service_area_id
JOIN merchant_outlets mo ON mo.location_id = l.location_id
WHERE sa.area_name = 'Tebet'
  AND sa.city = 'Jakarta'
  AND sa.area_status = 'ACTIVE'
  AND mo.outlet_status = 'OPEN'
ORDER BY mo.outlet_name, mo.merchant_outlet_id
LIMIT 20;
```

Dokumen outlet sudah menyimpan alamat dan koordinat lokasinya, jadi tidak perlu join ke `locations`. Tapi
nama dan status area ada di dokumen lain, jadi R10 dipecah jadi dua query.

Query pertama mencari area Tebet. Di sini dipakai rentang `_id` (`SAR...`) supaya CouchDB cuma cek 23
dokumen area. Tanpa rentang itu, CouchDB cek semua 460 ribu dokumen di `grab_geo` dan butuh sekitar 29 detik.
Query kedua mengambil 20 outlet pakai index `open-outlets-by-area`. Urutan `sort`-nya harus sama dengan
urutan field di index.

In [6]:
sc, ctx = start('PF-05-Q01')
areas = mango(GEO, json.loads('''
{
  "selector": {
    "_id": {"$gt": "SAR", "$lt": "SAS"},
    "type": "service_area",
    "city": "Jakarta",
    "area_name": "Tebet",
    "area_status": "ACTIVE"
  },
  "fields": ["_id", "area_name", "city", "area_status"],
  "limit": 1
}
'''))
display(pd.DataFrame(areas))

query = json.loads('''
{
  "selector": {
    "location.service_area_id": "SAR0000007",
    "outlet_status": "OPEN"
  },
  "sort": [
    {"location.service_area_id": "asc"},
    {"outlet_status": "asc"},
    {"outlet_name": "asc"},
    {"merchant_outlet_id": "asc"}
  ],
  "fields": ["merchant_outlet_id", "outlet_name", "location.address",
             "location.latitude", "location.longitude"],
  "limit": 20
}
''')
assert query['selector']['location.service_area_id'] == areas[0]['_id']
outlets = mango(GEO, query)
result = [{'merchant_outlet_id': d['merchant_outlet_id'], 'outlet_name': d['outlet_name'],
           'address': d['location']['address'], 'latitude': d['location']['latitude'],
           'longitude': d['location']['longitude']} for d in outlets]
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_geo/_find
-> 1 dokumen dalam 12.4 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 23


,_id,area_name,city,area_status
0,SAR0000007,Tebet,Jakarta,ACTIVE


POST /grab_geo/_find
-> 20 dokumen dalam 14.2 ms | index: _design/mango-outlets / open-outlets-by-area | dokumen diperiksa: 21


,merchant_outlet_id,outlet_name,address,latitude,longitude
0,OUT0026204,Ayam Geprek Adinata,"Jalan Dipatiukur No. 405, Tebet, Jakarta",-6.225576,106.842951
1,OUT0015214,Ayam Geprek Aditya,"Gg. Rajawali Barat No. 6, Tebet, Jakarta",-6.219450,106.849519
2,OUT0028367,Ayam Geprek Alika,"Gg. Bangka Raya No. 660, Tebet, Jakarta",-6.213585,106.847641
3,OUT0018928,Ayam Geprek Ana,"Gang Setiabudhi No. 68, Tebet, Jakarta",-6.240448,106.846137
4,OUT0013542,Ayam Geprek Ani,"Jalan Jamika No. 6, Tebet, Jakarta",-6.218772,106.862449



PF-05-Q01: lolos validasi = True.


## 2. Siklus perjalanan (W1–W5)

Semua write di bagian ini polanya mirip. Dokumennya dicari dulu pakai Mango, dan syarat `WHERE` di SQL
ditaruh di selector. Setelah itu dokumennya diubah di Python dan disimpan lagi pakai `PUT` dengan `_rev`
yang lama. Setiap penulisan dicek oleh `validate_doc_update`, yang fungsinya seperti trigger dan CHECK di
Part 2, jadi kalau transisi statusnya salah langsung ditolak.

Karena pesanan, payment, trip, dan assignment ada di satu dokumen, beberapa `INSERT` atau `UPDATE` di Part 2
biasanya cukup jadi satu `PUT`.

### PF-01-Q03 (W1): buat pesanan CONFIRMED dengan trip SEARCHING

SQL Part 2 (W1):
```sql
BEGIN;

INSERT INTO service_orders (order_id, created_at, order_status, total_amount, customer_id)
VALUES ('ORD9999001', CURRENT_TIMESTAMP, 'CONFIRMED', 15000.00, 'CUS0000001');

INSERT INTO mobility_orders (order_id, ride_type, passenger_count, pickup_notes)
VALUES ('ORD9999001', 'GrabBike', 1, NULL);

INSERT INTO payments (payment_id, payment_method, payment_status, amount, paid_at, order_id)
VALUES ('PAY9999001', 'CASH', 'PENDING', 15000.00, NULL, 'ORD9999001');

INSERT INTO trips (trip_id, started_at, completed_at, trip_status, order_id, start_location_id, end_location_id)
SELECT
    'TRP9999001',
    NULL,
    NULL,
    'SEARCHING',
    so.order_id,
    'LOC0000001',
    'LOC0000002'
FROM service_orders so
WHERE so.order_status = 'CONFIRMED'
  AND NOT EXISTS (SELECT 1 FROM trips t WHERE t.order_id = so.order_id)
LIMIT 1;

COMMIT;
```

CouchDB tidak punya foreign key, jadi customer dan lokasinya dicek dulu pakai Mango. Dua lokasi diambil
sekaligus pakai rentang `_id`. Empat `INSERT` di Part 2 jadi satu `PUT`.

In [7]:
sc, ctx = start('PF-01-Q03')
customer = mango(CUSTOMERS, json.loads('''
{
  "selector": {"_id": "CUS0000001"},
  "fields": ["_id"]
}
'''))
locations = mango(GEO, json.loads('''
{
  "selector": {
    "_id": {"$gte": "LOC0000001", "$lte": "LOC0000002"},
    "type": "location"
  }
}
'''))
assert customer and len(locations) == 2, 'customer atau lokasi tidak ada (pengganti foreign key)'
pickup, dropoff = locations

order = {
    '_id': 'ORD9999001', 'type': 'service_order', 'order_id': 'ORD9999001',
    'customer_id': 'CUS0000001', 'created_at': model.now(), 'order_status': 'CONFIRMED',
    'total_amount': 15000.0, 'service_type': 'MOBILITY',
    'mobility': {'ride_type': 'GrabBike', 'passenger_count': 1, 'pickup_notes': None},
    'payment': {'payment_id': 'PAY9999001', 'payment_method': 'CASH', 'payment_status': 'PENDING',
                'amount': 15000.0, 'paid_at': None},
    'trips': [new_trip('TRP9999001', pickup, dropoff)],
}
print(json.dumps(order, indent=2))
put(ORDERS, order)
finish(sc, ctx, order)

POST /grab_customers/_find
-> 1 dokumen dalam 4.8 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
POST /grab_geo/_find
-> 2 dokumen dalam 3.5 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 2
{
  "_id": "ORD9999001",
  "type": "service_order",
  "order_id": "ORD9999001",
  "customer_id": "CUS0000001",
  "created_at": "2026-11-23T20:36:02Z",
  "order_status": "CONFIRMED",
  "total_amount": 15000.0,
  "service_type": "MOBILITY",
  "mobility": {
    "ride_type": "GrabBike",
    "passenger_count": 1,
    "pickup_notes": null
  },
  "payment": {
    "payment_id": "PAY9999001",
    "payment_method": "CASH",
    "payment_status": "PENDING",
    "amount": 15000.0,
    "paid_at": null
  },
  "trips": [
    {
      "trip_id": "TRP9999001",
      "trip_status": "SEARCHING",
      "started_at": null,
      "completed_at": null,
      "start_location_id": "LOC0000001",
      "end_location_id": "LOC0000002",
      "start_service_area_id": "SAR0000021",
      "assignme

### PF-02-Q01 (W2): tawarkan trip ke driver yang tersedia

SQL Part 2 (W2):
```sql
INSERT INTO driver_assignments (assignment_id, assigned_at, assignment_status, responded_at, ended_at, end_reason, trip_id, vehicle_id)
SELECT 
    'ASN9999001',
    CURRENT_TIMESTAMP,
    'PENDING',
    NULL,
    NULL,
    NULL,
    'TRP9999001',
    v.vehicle_id
FROM vehicles v
WHERE v.driver_partner_id IS NOT NULL
LIMIT 1;
```

Data kendaraan ada di dalam dokumen driver, jadi driver yang ACTIVE dan punya motor ACTIVE dicari pakai
`$elemMatch`. Walaupun tanpa index, query ini tetap cepat karena ada `limit: 1`, jadi CouchDB berhenti begitu
ketemu driver pertama yang cocok. Setelah itu dokumen order dicari dengan syarat trip-nya masih SEARCHING,
lalu assignment PENDING ditambahkan.

In [8]:
sc, ctx = start('PF-02-Q01')
driver = mango(FLEET, json.loads('''
{
  "selector": {
    "type": "driver_partner",
    "driver_status": "ACTIVE",
    "vehicles": {"$elemMatch": {"vehicle_type": "Motorcycle", "vehicle_status": "ACTIVE"}}
  },
  "fields": ["_id", "vehicles"],
  "limit": 1
}
'''))[0]
vehicle = next(v for v in driver['vehicles']
               if v['vehicle_type'] == 'Motorcycle' and v['vehicle_status'] == 'ACTIVE')

order = mango(ORDERS, json.loads('''
{
  "selector": {
    "_id": "ORD9999001",
    "trips": {"$elemMatch": {"trip_id": "TRP9999001", "trip_status": "SEARCHING"}}
  }
}
'''))[0]
trip = find_trip(order, 'TRP9999001')
assert not any(a['assignment_status'] in ('PENDING', 'ACCEPTED') for a in trip['assignments']), \
    'trip sudah punya tawaran aktif'
trip['assignments'].append({
    'assignment_id': 'ASN9999001', 'assignment_status': 'PENDING', 'assigned_at': model.now(),
    'responded_at': None, 'ended_at': None, 'end_reason': None,
    'vehicle_id': vehicle['vehicle_id'], 'driver_partner_id': driver['_id'], 'vehicle_type': 'Motorcycle'})
put(ORDERS, order)
print(f"Ditawarkan ke {driver['_id']} ({vehicle['vehicle_id']})")
finish(sc, ctx, trip['assignments'][-1])

POST /grab_fleet/_find
-> 1 dokumen dalam 5.5 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 3
POST /grab_orders/_find
-> 1 dokumen dalam 3.2 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999001: _rev 1-e62366ac51adafcef7327cee1d406900 -> 2-46bc5c0666546ce2fa0b72c28dae282c
Ditawarkan ke DRV0000001 (VEH0000001)

PF-02-Q01: lolos validasi = True. Data uji sudah dihapus.


### PF-02-Q02 (W3): driver menerima tawaran dan mulai perjalanan

SQL Part 2 (W3):
```sql
BEGIN;

UPDATE driver_assignments
SET assignment_status = 'ACCEPTED', 
    responded_at = CURRENT_TIMESTAMP
WHERE assignment_id = 'ASN9999001';

UPDATE trips
SET trip_status = 'ALLOCATED'
WHERE trip_id = 'TRP9999001';

UPDATE trips
SET trip_status = 'IN_PROGRESS', 
    started_at = CURRENT_TIMESTAMP
WHERE trip_id = 'TRP9999001';

COMMIT;
```

`$elemMatch` di sini dua tingkat (trip, lalu assignment di dalamnya) untuk memastikan assignment-nya masih
PENDING. Dokumennya disimpan dua kali karena `validate_doc_update` mengecek setiap perubahan status: dari
SEARCHING ke ALLOCATED dulu, baru dari ALLOCATED ke IN_PROGRESS. Kalau langsung dari SEARCHING ke
IN_PROGRESS, CouchDB akan menolak.

In [9]:
sc, ctx = start('PF-02-Q02')
order = mango(ORDERS, json.loads('''
{
  "selector": {
    "_id": "ORD9999001",
    "trips": {"$elemMatch": {
      "assignments": {"$elemMatch": {"assignment_id": "ASN9999001", "assignment_status": "PENDING"}}
    }}
  }
}
'''))[0]
trip, assignment = find_assignment(order, 'ASN9999001')
assignment.update(assignment_status='ACCEPTED', responded_at=model.now())
trip['trip_status'] = 'ALLOCATED'
put(ORDERS, order)
trip.update(trip_status='IN_PROGRESS', started_at=model.now())
put(ORDERS, order)
finish(sc, ctx, trip)

POST /grab_orders/_find
-> 1 dokumen dalam 3.1 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999001: _rev 2-46bc5c0666546ce2fa0b72c28dae282c -> 3-1dc56a571f84fae76df8e2581dbaf175
PUT /grab_orders/ORD9999001: _rev 3-1dc56a571f84fae76df8e2581dbaf175 -> 4-888952a9632f2b341db6e40d9805128b

PF-02-Q02: lolos validasi = True. Data uji sudah dihapus.


### PF-02-Q03 (W4): driver menolak tawaran

SQL Part 2 (W4):
```sql
BEGIN;

INSERT INTO service_orders (order_id, created_at, order_status, total_amount, customer_id)
VALUES ('ORD9999002', CURRENT_TIMESTAMP, 'CONFIRMED', 15000.00, 'CUS0000001');

INSERT INTO mobility_orders (order_id, ride_type, passenger_count, pickup_notes)
VALUES ('ORD9999002', 'GrabBike', 1, NULL);

INSERT INTO payments (payment_id, payment_method, payment_status, amount, paid_at, order_id)
VALUES ('PAY9999002', 'CASH', 'PENDING', 15000.00, NULL, 'ORD9999002');

COMMIT;

INSERT INTO trips (trip_id, started_at, completed_at, trip_status, order_id, start_location_id, end_location_id)
SELECT 
    'TRP9999002',
    NULL,
    NULL,
    'SEARCHING',
    so.order_id,
    'LOC0000001',
    'LOC0000002'
FROM service_orders so
WHERE so.order_status = 'CONFIRMED'
  AND so.order_id <> 'ORD0000001'
  AND NOT EXISTS (SELECT 1 FROM trips t WHERE t.order_id = so.order_id)
LIMIT 1;

INSERT INTO driver_assignments (assignment_id, assigned_at, assignment_status, responded_at, ended_at, end_reason, trip_id, vehicle_id)
SELECT 
    'ASN9999002',
    CURRENT_TIMESTAMP,
    'PENDING',
    NULL,
    NULL,
    NULL,
    'TRP9999002',
    v.vehicle_id
FROM vehicles v
LIMIT 1 OFFSET 1;

UPDATE driver_assignments
SET assignment_status = 'REJECTED', 
    responded_at = CURRENT_TIMESTAMP, 
    ended_at = CURRENT_TIMESTAMP, 
    end_reason = 'Too far from pickup location'
WHERE assignment_id = 'ASN9999002';
```

Di Part 2, W4 juga membuat order baru dan menawarkannya dulu. Di sini bagian itu sudah disiapkan oleh data
uji (`ORD9999002` dengan assignment `ASN9999002`), jadi yang dijalankan cuma penolakannya. Trip-nya tetap
SEARCHING supaya bisa ditawarkan ke driver lain.

In [10]:
sc, ctx = start('PF-02-Q03')
order = mango(ORDERS, json.loads('''
{
  "selector": {
    "_id": "ORD9999002",
    "trips": {"$elemMatch": {
      "assignments": {"$elemMatch": {"assignment_id": "ASN9999002", "assignment_status": "PENDING"}}
    }}
  }
}
'''))[0]
trip, assignment = find_assignment(order, 'ASN9999002')
moment = model.now()
assignment.update(assignment_status='REJECTED', responded_at=moment, ended_at=moment,
                  end_reason='Too far from pickup location')
put(ORDERS, order)
finish(sc, ctx, assignment)

POST /grab_orders/_find
-> 1 dokumen dalam 3.3 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999002: _rev 2-8f7eacced3b724b88fec21e29461010c -> 3-9cb4ada756670c49bfd766cad2940c44

PF-02-Q03: lolos validasi = True. Data uji sudah dihapus.


### PF-03-Q02 (W5): selesaikan perjalanan dan bebaskan driver

SQL Part 2 (W5):
```sql
BEGIN;

UPDATE trips
SET trip_status = 'COMPLETED', 
    completed_at = CURRENT_TIMESTAMP
WHERE trip_id = 'TRP9999001';

UPDATE driver_assignments
SET ended_at = CURRENT_TIMESTAMP
WHERE trip_id = 'TRP9999001' 
  AND assignment_status = 'ACCEPTED';

UPDATE driver_partners
SET driver_status = 'ACTIVE'
WHERE driver_partner_id = (
    SELECT v.driver_partner_id 
    FROM vehicles v 
    JOIN driver_assignments da ON v.vehicle_id = da.vehicle_id 
    WHERE da.trip_id = 'TRP9999001' 
      AND da.assignment_status = 'ACCEPTED'
    LIMIT 1
);

COMMIT;
```

Yang berubah di sini ada dua dokumen di dua database (order dan driver), padahal CouchDB tidak punya
transaksi antardokumen. Jadi order disimpan dulu, baru driver. Update driver-nya selalu sama (jadi ACTIVE),
jadi kalau gagal tinggal diulang. `_id` driver baru ketahuan dari assignment yang ACCEPTED, jadi nilainya
diisi dari hasil query sebelumnya.

In [11]:
sc, ctx = start('PF-03-Q02')
order = mango(ORDERS, json.loads('''
{
  "selector": {
    "_id": "ORD9999001",
    "trips": {"$elemMatch": {"trip_id": "TRP9999001", "trip_status": "IN_PROGRESS"}}
  }
}
'''))[0]
trip = find_trip(order, 'TRP9999001')
moment = model.now()
trip.update(trip_status='COMPLETED', completed_at=moment)
accepted = [a for a in trip['assignments'] if a['assignment_status'] == 'ACCEPTED']
for assignment in accepted:
    assignment['ended_at'] = moment
put(ORDERS, order)

query = json.loads('''
{
  "selector": {"_id": "<driver_partner_id dari assignment ACCEPTED>"}
}
''')
query['selector']['_id'] = accepted[0]['driver_partner_id']
driver = mango(FLEET, query)[0]
driver['driver_status'] = 'ACTIVE'
put(FLEET, driver)
finish(sc, ctx, {'trip': trip, 'driver_partner_id': driver['_id'], 'driver_status': 'ACTIVE'})

POST /grab_orders/_find
-> 1 dokumen dalam 2.6 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999001: _rev 4-888952a9632f2b341db6e40d9805128b -> 5-731203e8bb8ebeaf7a199a6528844349
POST /grab_fleet/_find
-> 1 dokumen dalam 2.3 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_fleet/DRV0000001: _rev 16-474e7f4d336831d17f2076652c6a5c53 -> 17-f4ae292c69a824cc0fa158654a627959

PF-03-Q02: lolos validasi = True. Data uji sudah dihapus.


## 3. Pembayaran (W7–W9)

### PF-04-Q01 (W7): checkout non-tunai yang gagal sekali lalu berhasil

SQL Part 2 (W7):
```sql
BEGIN;

INSERT INTO service_orders (order_id, created_at, order_status, total_amount, customer_id)
VALUES ('ORD9999101', CURRENT_TIMESTAMP, 'PENDING', 20000.00, 'CUS0000001');

INSERT INTO mobility_orders (order_id, ride_type, passenger_count, pickup_notes)
VALUES ('ORD9999101', 'GrabCar', 2, 'Saya tunggu di lobi');

INSERT INTO payments (payment_id, payment_method, payment_status, amount, paid_at, order_id)
SELECT 'PAY9999101', 'E_WALLET', 'PENDING', so.total_amount, NULL, so.order_id
FROM service_orders so
WHERE so.order_id = 'ORD9999101';

COMMIT;

UPDATE payments
SET payment_status = 'FAILED'
WHERE payment_id = 'PAY9999101'
  AND payment_status = 'PENDING';

BEGIN;

UPDATE payments
SET payment_status = 'PENDING'
WHERE payment_id = 'PAY9999101'
  AND payment_status = 'FAILED';

UPDATE payments
SET payment_status = 'PAID', paid_at = CURRENT_TIMESTAMP
WHERE payment_id = 'PAY9999101'
  AND payment_status = 'PENDING';

UPDATE service_orders so
SET order_status = 'CONFIRMED'
FROM payments p
WHERE p.order_id = so.order_id
  AND so.order_id = 'ORD9999101'
  AND so.order_status = 'PENDING'
  AND p.payment_status = 'PAID';

INSERT INTO trips (trip_id, started_at, completed_at, trip_status, order_id, start_location_id, end_location_id)
SELECT 'TRP9999101', NULL, NULL, 'SEARCHING', so.order_id, 'LOC9999101', 'LOC9999102'
FROM service_orders so
WHERE so.order_id = 'ORD9999101'
  AND so.order_status = 'CONFIRMED';

COMMIT;
```

Setiap perubahan status payment disimpan sebagai satu revisi, supaya `validate_doc_update` bisa mengecek
setiap transisinya (PENDING, FAILED, PENDING, lalu PAID). Di akhir, dokumennya ada di revisi ke-4 (`_rev`
diawali `4-`).

In [12]:
sc, ctx = start('PF-04-Q01')
customer = mango(CUSTOMERS, json.loads('''
{
  "selector": {"_id": "CUS0000001"},
  "fields": ["_id"]
}
'''))
locations = mango(GEO, json.loads('''
{
  "selector": {
    "_id": {"$gte": "LOC0000001", "$lte": "LOC0000002"},
    "type": "location"
  }
}
'''))
assert customer and len(locations) == 2
pickup, dropoff = locations

order = {
    '_id': 'ORD9999101', 'type': 'service_order', 'order_id': 'ORD9999101',
    'customer_id': 'CUS0000001', 'created_at': model.now(), 'order_status': 'PENDING',
    'total_amount': 20000.0, 'service_type': 'MOBILITY',
    'mobility': {'ride_type': 'GrabCar', 'passenger_count': 2, 'pickup_notes': 'Saya tunggu di lobi'},
    'payment': {'payment_id': 'PAY9999101', 'payment_method': 'E_WALLET', 'payment_status': 'PENDING',
                'amount': 20000.0, 'paid_at': None},
    'trips': [],
}
put(ORDERS, order)                                    # order dibuat, payment PENDING
order['payment']['payment_status'] = 'FAILED'         # gateway menolak
put(ORDERS, order)
order['payment']['payment_status'] = 'PENDING'        # customer mencoba lagi
put(ORDERS, order)
order['payment'].update(payment_status='PAID', paid_at=model.now())
order['order_status'] = 'CONFIRMED'
order['trips'].append(new_trip('TRP9999101', pickup, dropoff))
put(ORDERS, order)
finish(sc, ctx, order)

POST /grab_customers/_find
-> 1 dokumen dalam 2.5 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
POST /grab_geo/_find
-> 2 dokumen dalam 2.6 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 2
PUT /grab_orders/ORD9999101: _rev (dokumen baru) -> 1-a9f735825eabb8c08d870a9324997145
PUT /grab_orders/ORD9999101: _rev 1-a9f735825eabb8c08d870a9324997145 -> 2-3d69cfb02f3da2035ec59b3d1b422332
PUT /grab_orders/ORD9999101: _rev 2-3d69cfb02f3da2035ec59b3d1b422332 -> 3-6e643ac571841345a007046d81536807
PUT /grab_orders/ORD9999101: _rev 3-6e643ac571841345a007046d81536807 -> 4-bdc32ac9940f541929c0b7bd8a02440b

PF-04-Q01: lolos validasi = True. Data uji sudah dihapus.


### PF-04-Q02 (W8): customer membatalkan pesanan sebelum dijemput

SQL Part 2 (W8):
```sql
BEGIN;

UPDATE service_orders
SET order_status = 'CANCELLED'
WHERE order_id = 'ORD9999101'
  AND order_status = 'CONFIRMED';

UPDATE trips t
SET trip_status = 'CANCELLED'
FROM service_orders so
WHERE so.order_id = t.order_id
  AND so.order_id = 'ORD9999101'
  AND so.order_status = 'CANCELLED'
  AND t.trip_status IN ('SEARCHING', 'ALLOCATED');

UPDATE driver_assignments da
SET assignment_status = 'CANCELLED',
    ended_at = CURRENT_TIMESTAMP,
    end_reason = 'Cancelled by customer'
FROM trips t
WHERE t.trip_id = da.trip_id
  AND t.order_id = 'ORD9999101'
  AND t.trip_status = 'CANCELLED'
  AND da.assignment_status IN ('PENDING', 'ACCEPTED');

UPDATE payments p
SET payment_status = 'REFUNDED'
FROM service_orders so
WHERE so.order_id = p.order_id
  AND so.order_id = 'ORD9999101'
  AND so.order_status = 'CANCELLED'
  AND p.payment_status = 'PAID';

UPDATE payments p
SET payment_status = 'FAILED'
FROM service_orders so
WHERE so.order_id = p.order_id
  AND so.order_id = 'ORD9999101'
  AND so.order_status = 'CANCELLED'
  AND p.payment_status = 'PENDING';

COMMIT;
```

Syarat `order_status: CONFIRMED` di selector fungsinya sama dengan `WHERE order_status = 'CONFIRMED'`. Kalau
ordernya sudah jalan, sudah selesai, atau sudah dibatalkan, hasilnya kosong dan tidak ada yang diubah.
Order, trip, assignment, dan payment berubah sekaligus dalam satu `PUT`.

In [13]:
sc, ctx = start('PF-04-Q02')
found = mango(ORDERS, json.loads('''
{
  "selector": {"_id": "ORD9999101", "order_status": "CONFIRMED"}
}
'''))
cancelled = bool(found)
if cancelled:
    order = found[0]
    moment = model.now()
    order['order_status'] = 'CANCELLED'
    for trip in order['trips']:
        if trip['trip_status'] in ('SEARCHING', 'ALLOCATED'):
            trip['trip_status'] = 'CANCELLED'
            for a in trip['assignments']:
                if a['assignment_status'] in ('PENDING', 'ACCEPTED'):
                    a.update(assignment_status='CANCELLED', ended_at=moment,
                             end_reason='Cancelled by customer')
    payment = order['payment']
    if payment['payment_status'] == 'PAID':
        payment['payment_status'] = 'REFUNDED'       # paid_at tetap
    elif payment['payment_status'] == 'PENDING':
        payment['payment_status'] = 'FAILED'
    put(ORDERS, order)
    print(f"order CANCELLED, payment {payment['payment_status']}")
finish(sc, ctx, {'cancelled': cancelled, 'order': found[0] if found else None})

POST /grab_orders/_find
-> 1 dokumen dalam 3.2 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999101: _rev 5-007bbeea0c912e7a3c27cc3185f12d8f -> 6-f1e7b680a5d6ce7100e9d991132474bb
order CANCELLED, payment REFUNDED

PF-04-Q02: lolos validasi = True. Data uji sudah dihapus.


### PF-04-Q03 (W9): selesaikan ride tunai dan terima pembayarannya

SQL Part 2 (W9):
```sql
BEGIN;

INSERT INTO service_orders (order_id, created_at, order_status, total_amount, customer_id)
VALUES ('ORD9999102', CURRENT_TIMESTAMP - INTERVAL '30 minutes', 'IN_PROGRESS', 18000.00, 'CUS0000001');

INSERT INTO mobility_orders (order_id, ride_type, passenger_count, pickup_notes)
VALUES ('ORD9999102', 'GrabBike', 1, NULL);

INSERT INTO payments (payment_id, payment_method, payment_status, amount, paid_at, order_id)
VALUES ('PAY9999102', 'CASH', 'PENDING', 18000.00, NULL, 'ORD9999102');

INSERT INTO trips (trip_id, started_at, completed_at, trip_status, order_id, start_location_id, end_location_id)
VALUES ('TRP9999102', CURRENT_TIMESTAMP - INTERVAL '20 minutes', NULL, 'IN_PROGRESS', 'ORD9999102',
        'LOC9999102', 'LOC9999103');

COMMIT;

BEGIN;

UPDATE trips
SET trip_status = 'COMPLETED', completed_at = CURRENT_TIMESTAMP
WHERE trip_id = 'TRP9999102';

UPDATE service_orders
SET order_status = 'COMPLETED'
WHERE order_id = 'ORD9999102';

UPDATE payments
SET payment_status = 'PAID', paid_at = CURRENT_TIMESTAMP
WHERE order_id = 'ORD9999102'
  AND payment_method = 'CASH';

COMMIT;
```

Transaksi pertama W9 di Part 2 menyiapkan ride tunai yang sedang jalan, dan di sini ride itu sudah
disiapkan oleh data uji (`ORD9999102`). Trip, order, dan payment berubah dalam satu `PUT`, jadi tidak mungkin
ride tercatat selesai tapi pembayarannya belum.

In [14]:
sc, ctx = start('PF-04-Q03')
order = mango(ORDERS, json.loads('''
{
  "selector": {
    "_id": "ORD9999102",
    "payment.payment_method": "CASH",
    "trips": {"$elemMatch": {"trip_id": "TRP9999102", "trip_status": "IN_PROGRESS"}}
  }
}
'''))[0]
moment = model.now()
find_trip(order, 'TRP9999102').update(trip_status='COMPLETED', completed_at=moment)
order['order_status'] = 'COMPLETED'
order['payment'].update(payment_status='PAID', paid_at=moment)
put(ORDERS, order)
finish(sc, ctx, order)

POST /grab_orders/_find
-> 1 dokumen dalam 2.9 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_orders/ORD9999102: _rev 1-709a528beb1bf38aa317aeecc6bcbaa6 -> 2-3c43060b37a429831269b99fdf495477

PF-04-Q03: lolos validasi = True. Data uji sudah dihapus.


## 4. Service area dan outlet (W6, W10, PF-07)

### PF-06-Q01 (W6): daftarkan service area baru beserta lokasi dan outletnya

SQL Part 2 (W6):
```sql
BEGIN;

INSERT INTO service_areas (service_area_id, area_name, city, area_status)
VALUES ('SAR9999101', 'Cicendo', 'Bandung', 'ACTIVE');

INSERT INTO locations (location_id, latitude, longitude, address, service_area_id)
VALUES
    ('LOC9999101', -6.905200, 107.596400, 'Jl. Pajajaran No. 12, Cicendo, Bandung', 'SAR9999101'),
    ('LOC9999102', -6.911800, 107.601700, 'Jl. Kebon Kawung No. 5, Cicendo, Bandung', 'SAR9999101'),
    ('LOC9999103', -6.899600, 107.589300, 'Jl. Pasirkaliki No. 88, Cicendo, Bandung', 'SAR9999101');

INSERT INTO merchant_outlets (merchant_outlet_id, outlet_name, outlet_status, location_id)
VALUES
    ('OUT9999101', 'Warung Bu Sri', 'OPEN', 'LOC9999101'),
    ('OUT9999102', 'Kedai Kopi Pak Dadang', 'OPEN', 'LOC9999103');

COMMIT;
```

Pertama dicek dulu pakai Mango apakah Cicendo, Bandung sudah ada. Setelah itu area, 3 lokasi, dan 2 outlet
dikirim sekaligus pakai `_bulk_docs`. Bedanya dengan transaksi SQL, `_bulk_docs` tidak all-or-nothing, jadi
bisa saja sebagian dokumen gagal. Kalau itu terjadi, dokumen yang sudah masuk dihapus lagi.

In [15]:
sc, ctx = start('PF-06-Q01')
same_name = mango(GEO, json.loads('''
{
  "selector": {
    "_id": {"$gt": "SAR", "$lt": "SAS"},
    "city": "Bandung",
    "area_name": "Cicendo"
  },
  "fields": ["_id"]
}
'''))
assert not same_name, 'Cicendo, Bandung sudah ada'

area = model.service_area_doc(adapter.W6_AREA)
locations = [model.location_doc({**loc, 'service_area_id': area['_id']}) for loc in adapter.W6_LOCATIONS]
by_id = {loc['_id']: loc for loc in locations}
outlets = [model.outlet_doc(o, by_id[o['location_id']]) for o in adapter.W6_OUTLETS]
docs = [area, *locations, *outlets]

print(f'POST /{GEO}/_bulk_docs ({len(docs)} dokumen)')
results = app.bulk_docs(GEO, docs)
failed = [r for r in results if 'error' in r]
if failed:
    written = [{'_id': r['id'], '_rev': r['rev'], '_deleted': True} for r in results if 'rev' in r]
    app.bulk_docs(GEO, written)
    raise RuntimeError(f'Onboarding dibatalkan: {failed[0]}')
display(pd.DataFrame(results))
finish(sc, ctx, {'service_area_id': area['_id'], 'documents_written': len(results)})

POST /grab_geo/_find
-> 0 dokumen dalam 4.5 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 23
POST /grab_geo/_bulk_docs (6 dokumen)


,ok,id,rev
0,True,SAR9999101,1-2805b0bf65785ff8a6126c1147e9be5f
1,True,LOC9999101,1-9cfaa4cfb23066c6cd997778814c8899
2,True,LOC9999102,1-7947c025aa12336b3ce365f3730b664a
3,True,LOC9999103,1-f8c1721ad63043899fcd00c56e8b6251
4,True,OUT9999101,1-fc0e59580d259950826ce406cfb31143
5,True,OUT9999102,1-01c9a4c80afba587c800a6fff0d1460a



PF-06-Q01: lolos validasi = True. Data uji sudah dihapus.


### PF-06-Q02 (W10): nonaktifkan service area dan delist outletnya

SQL Part 2 (W10):
```sql
BEGIN;

UPDATE service_areas
SET area_status = 'INACTIVE'
WHERE service_area_id = 'SAR9999101'
  AND area_status = 'ACTIVE';

UPDATE merchant_outlets mo
SET outlet_status = 'INACTIVE'
FROM locations l
WHERE mo.location_id = l.location_id
  AND l.service_area_id = 'SAR9999101'
  AND mo.outlet_status <> 'INACTIVE'
  AND mo.merchant_outlet_id NOT IN (
      SELECT f.merchant_outlet_id
      FROM food_orders f
      JOIN service_orders so ON so.order_id = f.order_id
      WHERE so.order_status IN ('PENDING', 'CONFIRMED', 'IN_PROGRESS')
  );

UPDATE merchant_outlets mo
SET outlet_status = 'CLOSED'
FROM locations l
WHERE mo.location_id = l.location_id
  AND l.service_area_id = 'SAR9999101'
  AND mo.outlet_status = 'OPEN';

COMMIT;
```

Area yang masih ACTIVE dicari pakai Mango, lalu diubah jadi INACTIVE. Setelah itu outlet di area tersebut
yang belum INACTIVE dicari, juga pakai Mango. Ada satu hal yang perlu diperhatikan di sini: index Mango baru
dipakai kalau semua field index-nya ada di selector. Index `open-outlets-by-area` punya 4 field, jadi 2 field
sisanya ditambahkan dengan `$exists: true`. Tanpa itu, CouchDB cek 460 ribu dokumen (sekitar 25 detik),
padahal yang dicari cuma 2.

Untuk cek apakah outlet masih punya order yang berjalan (bagian `NOT IN (SELECT ...)` di SQL), query
Mango-nya harus scan 2 juta order dan butuh sekitar 130 detik. Jadi yang ditampilkan cuma `_explain`-nya,
dan jawabannya diambil dari view `ops/food_by_outlet`. Terakhir, outlet yang tidak punya order berjalan jadi
INACTIVE, sedangkan yang masih punya order berjalan cukup jadi CLOSED. Semuanya disimpan pakai satu
`_bulk_docs`.

In [16]:
sc, ctx = start('PF-06-Q02')
area = mango(GEO, json.loads('''
{
  "selector": {"_id": "SAR9999101", "area_status": "ACTIVE"}
}
'''))[0]
area['area_status'] = 'INACTIVE'
put(GEO, area)

outlets = mango(GEO, json.loads('''
{
  "selector": {
    "location.service_area_id": "SAR9999101",
    "outlet_status": {"$in": ["OPEN", "CLOSED"]},
    "outlet_name": {"$exists": true},
    "merchant_outlet_id": {"$exists": true}
  }
}
'''))
outlet_ids = [o['_id'] for o in outlets]

running_query = json.loads('''
{
  "selector": {
    "type": "service_order",
    "service_type": "FOOD",
    "food.merchant_outlet_id": {"$in": ["OUT9999102", "OUT9999101"]},
    "order_status": {"$in": ["PENDING", "CONFIRMED", "IN_PROGRESS"]}
  },
  "fields": ["_id", "order_status", "food.merchant_outlet_id"]
}
''')
assert sorted(running_query['selector']['food.merchant_outlet_id']['$in']) == sorted(outlet_ids)
explain(ORDERS, running_query)
statuses = running_query['selector']['order_status']['$in']
rows = timed(lambda: app.view(ORDERS, 'ops', 'food_by_outlet', group=True,
                              keys=[[o, s] for o in outlet_ids for s in statuses])['rows'],
             'view ops/food_by_outlet')
running = {row['key'][0] for row in rows if row['value']}
print('Outlet dengan food order berjalan:', sorted(running))

changed = []
for outlet in outlets:
    if outlet['outlet_status'] != 'INACTIVE' and outlet['_id'] not in running:
        outlet['outlet_status'] = 'INACTIVE'
        changed.append(outlet)
    elif outlet['outlet_status'] == 'OPEN':
        outlet['outlet_status'] = 'CLOSED'
        changed.append(outlet)
print(f'POST /{GEO}/_bulk_docs')
display(pd.DataFrame(app.bulk_docs(GEO, changed)))
display(pd.DataFrame([{'_id': o['_id'], 'outlet_status': o['outlet_status']} for o in outlets]))
finish(sc, ctx, None)

POST /grab_geo/_find
-> 1 dokumen dalam 2.7 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
PUT /grab_geo/SAR9999101: _rev 1-2805b0bf65785ff8a6126c1147e9be5f -> 2-bec3ecec3d0314c7a32cb86334b842df
POST /grab_geo/_find
-> 2 dokumen dalam 5.7 ms | index: _design/mango-outlets / open-outlets-by-area | dokumen diperiksa: 2
POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view ops/food_by_outlet: 17.4 ms
Outlet dengan food order berjalan: ['OUT9999102']
POST /grab_geo/_bulk_docs


,ok,id,rev
0,True,OUT9999102,2-4bfbcd7802baa71013696de4c02e9ee2
1,True,OUT9999101,2-708a2d0619a5e2f06c8ba39531dc10b9


,_id,outlet_status
0,OUT9999102,CLOSED
1,OUT9999101,INACTIVE



PF-06-Q02: lolos validasi = True. Data uji sudah dihapus.


### PF-07-Q01 dan PF-07-Q02: hapus outlet

Ini juga tidak ada di Part 2, karena di sana penghapusan sudah dijaga foreign key. Di CouchDB, sebelum
`DELETE` aplikasi harus cek sendiri dua hal: outletnya sudah INACTIVE, dan tidak ada food order yang masih
merujuk ke outlet itu. Cek yang kedua sama seperti di W10, jadi pakai view `ops/food_by_outlet`.

Di PF-07-Q01, `OUT9999201` sudah INACTIVE dan tidak dipakai order mana pun, jadi bisa dihapus. Di PF-07-Q02,
`OUT9999202` masih dipakai satu order, jadi penghapusannya ditolak.

In [17]:
def delete_outlet(outlet_id):
    query = json.loads('''
    {
      "selector": {"_id": "<outlet_id>", "type": "merchant_outlet"}
    }
    ''')
    query['selector']['_id'] = outlet_id
    outlet = mango(GEO, query)[0]
    references_query = json.loads('''
    {
      "selector": {
        "type": "service_order",
        "service_type": "FOOD",
        "food.merchant_outlet_id": "<outlet_id>"
      },
      "fields": ["_id"]
    }
    ''')
    references_query['selector']['food.merchant_outlet_id'] = outlet_id
    explain(ORDERS, references_query)
    rows = app.view(ORDERS, 'ops', 'food_by_outlet', startkey=[outlet_id], endkey=[outlet_id, {}])['rows']
    references = int(rows[0]['value']) if rows else 0
    print(f"{outlet_id}: {outlet['outlet_status']}, dirujuk {references} food order (view ops/food_by_outlet)")
    if outlet['outlet_status'] != 'INACTIVE':
        return {'deleted': False, 'reason': f"outlet is {outlet['outlet_status']}", 'food_orders': references}
    if references:
        return {'deleted': False, 'reason': 'outlet is referenced by food orders', 'food_orders': references}
    rev = app.delete_doc(GEO, outlet_id, outlet['_rev'])
    print(f'DELETE /{GEO}/{outlet_id}?rev={outlet["_rev"]}')
    return {'deleted': True, 'rev': rev, 'food_orders': 0}


for operation_id, outlet_id in [('PF-07-Q01', 'OUT9999201'), ('PF-07-Q02', 'OUT9999202')]:
    print(f'=== {operation_id}')
    sc, ctx = start(operation_id)
    result = delete_outlet(outlet_id)
    print(result)
    finish(sc, ctx, result)
    print()

=== PF-07-Q01
POST /grab_geo/_find
-> 1 dokumen dalam 3.1 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
OUT9999201: INACTIVE, dirujuk 0 food order (view ops/food_by_outlet)
DELETE /grab_geo/OUT9999201?rev=1-0f105178111399300f0d2d4cec1f0026
{'deleted': True, 'rev': '2-2f58a2926eeb2270ab10d8efcd657129', 'food_orders': 0}

PF-07-Q01: lolos validasi = True. Data uji sudah dihapus.

=== PF-07-Q02
POST /grab_geo/_find
-> 1 dokumen dalam 2.4 ms | index: tanpa index Mango (_all_docs) | dokumen diperiksa: 1
POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
OUT9999202: INACTIVE, dirujuk 1 food order (view ops/food_by_outlet)
{'deleted': False, 'reason': 'outlet is referenced by food orders', 'food_orders': 1}

PF-07-Q02: lolos validasi = True. Data uji sudah dihapus.



## 5. Analitik (R1–R9)

Semua query di bagian ini butuh agregasi (`GROUP BY`, `COUNT`, `SUM`), sedangkan Mango cuma bisa memilih
dokumen. Untuk setiap query tetap ditulis query Mango yang memilih dokumen sumbernya, lalu dicek
`_explain`-nya. Hasilnya selalu sama: CouchDB harus scan 2 juta order, dan datanya masih harus ditarik ke
Python untuk dijumlahkan. Makanya implementasinya pakai view map/reduce, yang menghitung agregatnya sekali
waktu index dibangun, lalu tinggal diperbarui sedikit-sedikit kalau ada data baru. Hasil setiap query dicek
dengan hasil acuan dari SQL Part 2.

### PF-08-Q01 (R1): Tingkat penerimaan penugasan per mitra pengemudi

SQL Part 2 (R1):
```sql
SELECT 
    dp.driver_partner_id,
    dp.full_name,
    COUNT(CASE WHEN da.assignment_status = 'ACCEPTED' THEN 1 END) AS total_accepted,
    COUNT(CASE WHEN da.assignment_status = 'REJECTED' THEN 1 END) AS total_rejected,
    COUNT(CASE WHEN da.assignment_status = 'EXPIRED' THEN 1 END) AS total_expired,
    ROUND(
        (COUNT(CASE WHEN da.assignment_status = 'ACCEPTED' THEN 1 END)::NUMERIC / 
        NULLIF(COUNT(CASE WHEN da.assignment_status IN ('ACCEPTED', 'REJECTED') THEN 1 END), 0)) * 100, 2
    ) AS acceptance_rate_percentage
FROM driver_partners dp
JOIN vehicles v ON dp.driver_partner_id = v.driver_partner_id
JOIN driver_assignments da ON v.vehicle_id = da.vehicle_id
GROUP BY dp.driver_partner_id, dp.full_name
HAVING COUNT(CASE WHEN da.assignment_status IN ('ACCEPTED', 'REJECTED') THEN 1 END) >= 5
ORDER BY acceptance_rate_percentage DESC, total_accepted DESC;
```

Di sini perlu hitungan per driver dari semua assignment, lalu nama driver diambil dari `grab_fleet`. Dua-duanya tidak bisa dilakukan Mango.

In [18]:
sc, ctx = start('PF-08-Q01')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {
      "assignments": {"$elemMatch": {"assignment_status": {"$in": ["ACCEPTED", "REJECTED", "EXPIRED"]}}}
    }}
  },
  "fields": ["trips"]
}
'''))
result = timed(lambda: grab.driver_acceptance_rates(**ctx['params']), 'view analytics/driver_decisions + fleet/driver_names')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/driver_decisions + fleet/driver_names: 14,673.7 ms


,driver_partner_id,full_name,total_accepted,total_rejected,total_expired,acceptance_rate_percentage
0,DRV0071552,Padma Laksita,28,0,4,100.0
1,DRV0068730,Jamal Sihotang,25,0,7,100.0
2,DRV0099370,Cahyanto Winarno,24,0,7,100.0
3,DRV0016513,Pranata Puspita,23,0,4,100.0
4,DRV0057417,Hari Ardianto,23,0,2,100.0



PF-08-Q01: lolos validasi = True.


### PF-08-Q02 (R2): 10 mitra pengemudi dengan total durasi perjalanan selesai terbesar

SQL Part 2 (R2):
```sql
SELECT 
    dp.driver_partner_id,
    dp.full_name,
    COUNT(t.trip_id) AS total_completed_trips,
    ROUND(SUM(EXTRACT(EPOCH FROM (t.completed_at - t.started_at)) / 60)::NUMERIC, 2) AS total_duration_minutes,
    ROUND(AVG(EXTRACT(EPOCH FROM (t.completed_at - t.started_at)) / 60)::NUMERIC, 2) AS avg_duration_per_trip_minutes
FROM driver_partners dp
JOIN vehicles v ON dp.driver_partner_id = v.driver_partner_id
JOIN driver_assignments da ON v.vehicle_id = da.vehicle_id
JOIN trips t ON da.trip_id = t.trip_id
WHERE da.assignment_status = 'ACCEPTED'
  AND t.trip_status = 'COMPLETED'
GROUP BY dp.driver_partner_id, dp.full_name
ORDER BY total_duration_minutes DESC
LIMIT 10;
```

Butuh total durasi per driver (`SUM`) lalu diambil 10 teratas, dan Mango tidak bisa menjumlahkan.

In [19]:
sc, ctx = start('PF-08-Q02')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {"trip_status": "COMPLETED"}}
  },
  "fields": ["trips"]
}
'''))
result = timed(lambda: grab.top_drivers_by_duration(**ctx['params']), 'view analytics/driver_completed_duration')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/driver_completed_duration: 8,219.2 ms


,driver_partner_id,full_name,total_completed_trips,total_duration_minutes,avg_duration_per_trip_minutes
0,DRV0064085,Kacung Wulandari,51,1376.90,27.00
1,DRV0030293,Harjasa Habibi,62,1375.73,22.19
2,DRV0075421,Lala Laksita,57,1367.82,24.00
3,DRV0040086,Kawaca Rahmawati,58,1324.55,22.84
4,DRV0069666,Adika Pradana,50,1308.90,26.18



PF-08-Q02: lolos validasi = True.


### PF-08-Q03 (R3): Jumlah perjalanan dan kendaraan berbeda per tipe kendaraan dan jam

SQL Part 2 (R3):
```sql
SELECT 
    v.vehicle_type,
    EXTRACT(HOUR FROM t.started_at) AS trip_hour,
    COUNT(t.trip_id) AS total_trips,
    COUNT(DISTINCT v.vehicle_id) AS unique_vehicles_used
FROM vehicles v
JOIN driver_assignments da ON v.vehicle_id = da.vehicle_id
JOIN trips t ON da.trip_id = t.trip_id
WHERE da.assignment_status = 'ACCEPTED'
  AND t.trip_status IN ('COMPLETED', 'IN_PROGRESS')
GROUP BY v.vehicle_type, EXTRACT(HOUR FROM t.started_at)
ORDER BY trip_hour ASC, total_trips DESC;
```

Butuh `COUNT` dan `COUNT(DISTINCT vehicle_id)` per tipe kendaraan dan jam, jadi tetap harus pakai view.

In [20]:
sc, ctx = start('PF-08-Q03')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {"trip_status": {"$in": ["IN_PROGRESS", "COMPLETED"]}}}
  },
  "fields": ["trips"]
}
'''))
result = timed(lambda: grab.vehicle_type_hourly_utilization(**ctx['params']), 'view analytics/vehicle_hourly_usage')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/vehicle_hourly_usage: 12,299.5 ms


,vehicle_type,trip_hour,total_trips,unique_vehicles_used
0,Motorcycle,0,49069,33561
1,Car,0,12350,9925
2,Electric Motorcycle,0,4502,3121
3,Premium Car,0,2228,1958
4,Electric Car,0,1405,1202



PF-08-Q03: lolos validasi = True.


### PF-02-Q04 (R4): Perjalanan yang ditawarkan lebih dari dua kali lalu diterima

SQL Part 2 (R4):
```sql
WITH trip_assignment_counts AS (
    SELECT 
        trip_id,
        COUNT(assignment_id) AS total_attempts,
        COUNT(CASE WHEN assignment_status = 'ACCEPTED' THEN 1 END) AS has_accepted
    FROM driver_assignments
    GROUP BY trip_id
)
SELECT 
    t.trip_id,
    t.order_id,
    t.trip_status,
    tac.total_attempts,
    t.started_at,
    t.completed_at
FROM trips t
JOIN trip_assignment_counts tac ON t.trip_id = tac.trip_id
WHERE tac.total_attempts > 2
  AND tac.has_accepted > 0
ORDER BY tac.total_attempts DESC, t.started_at DESC;
```

Selain butuh hitungan, syarat "minimal 3 tawaran" juga tidak bisa ditulis di Mango. `$size` cuma bisa mencocokkan panjang array yang persis sama, tidak bisa "lebih dari".

In [21]:
sc, ctx = start('PF-02-Q04')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {
      "assignments": {"$elemMatch": {"assignment_status": "ACCEPTED"}}
    }}
  },
  "fields": ["_id", "trips"]
}
'''))
result = timed(lambda: grab.reassignment_history(**ctx['params']), 'view analytics/trip_attempts')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/trip_attempts: 5,682.0 ms


,trip_id,order_id,trip_status,total_attempts,started_at,completed_at
0,TRP1956930,ORD1998815,ALLOCATED,4,NaN,NaN
1,TRP1956843,ORD1998726,ALLOCATED,4,NaN,NaN
2,TRP1956771,ORD1998653,ALLOCATED,4,NaN,NaN
3,TRP1956764,ORD1998645,ALLOCATED,4,NaN,NaN
4,TRP1956685,ORD1998563,ALLOCATED,4,NaN,NaN



PF-02-Q04: lolos validasi = True.


### PF-09-Q01 (R5): Jumlah perjalanan per wilayah layanan

SQL Part 2 (R5):
```sql
SELECT 
    sa.service_area_id,
    sa.area_name,
    sa.city,
    COUNT(t.trip_id) AS total_trips
FROM service_areas sa
JOIN locations l ON sa.service_area_id = l.service_area_id
JOIN trips t ON l.location_id = t.start_location_id
GROUP BY sa.service_area_id, sa.area_name, sa.city
ORDER BY total_trips DESC;
```

Cuma perlu `COUNT` per service area, tapi itu pun tidak bisa di Mango.

In [22]:
sc, ctx = start('PF-09-Q01')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "trips": {"$elemMatch": {"trip_id": {"$exists": true}}}
  },
  "fields": ["trips"]
}
'''))
result = timed(lambda: grab.trips_per_service_area(**ctx['params']), 'view analytics/trips_by_start_area')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/trips_by_start_area: 82.9 ms


,service_area_id,area_name,city,total_trips
0,SAR0000009,Coblong,Bandung,86234
1,SAR0000010,Sukajadi,Bandung,85960
2,SAR0000005,Cengkareng,Jakarta,85911
3,SAR0000001,Menteng,Jakarta,85892
4,SAR0000012,Buahbatu,Bandung,85881



PF-09-Q01: lolos validasi = True.


### PF-10-Q01 (R6): Pendapatan bersih dan pengembalian dana per wilayah

SQL Part 2 (R6):
```sql
WITH first_trip AS (
    SELECT order_id, MIN(trip_id) AS trip_id
    FROM trips
    GROUP BY order_id
)
SELECT
    sa.city,
    sa.area_name,
    COUNT(*) AS settled_payments,
    COALESCE(SUM(p.amount) FILTER (WHERE p.payment_status = 'PAID'), 0) AS net_revenue,
    COALESCE(SUM(p.amount) FILTER (WHERE p.payment_status = 'REFUNDED'), 0) AS refunded_amount,
    ROUND(100.0 * COUNT(*) FILTER (WHERE p.payment_status = 'REFUNDED') / COUNT(*), 2) AS refund_rate_pct
FROM payments p
JOIN first_trip ft ON ft.order_id = p.order_id
JOIN trips t ON t.trip_id = ft.trip_id
JOIN locations l ON l.location_id = t.start_location_id
JOIN service_areas sa ON sa.service_area_id = l.service_area_id
WHERE p.payment_status IN ('PAID', 'REFUNDED')
GROUP BY sa.service_area_id, sa.city, sa.area_name
ORDER BY net_revenue DESC;
```

Butuh `SUM` pembayaran per service area dari trip pertama setiap order.

In [23]:
sc, ctx = start('PF-10-Q01')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "payment.payment_status": {"$in": ["PAID", "REFUNDED"]}
  },
  "fields": ["payment", "trips"]
}
'''))
result = timed(lambda: grab.revenue_and_refunds_per_area(**ctx['params']), 'view analytics/settled_payments_by_area')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/settled_payments_by_area: 108.8 ms


,city,area_name,settled_payments,net_revenue,refunded_amount,refund_rate_pct
0,Jakarta,Cengkareng,82447,5.536849e+09,555359500.0,9.16
1,Jakarta,Kelapa Gading,82103,5.305748e+09,543239500.0,9.28
2,Jakarta,Cilandak,81172,4.497901e+09,454400000.0,9.09
3,Jakarta,Pasar Minggu,81213,4.401596e+09,450875000.0,9.30
4,Jakarta,Menteng,82511,3.902126e+09,398626000.0,9.26



PF-10-Q01: lolos validasi = True.


### PF-10-Q02 (R7): Ringkasan pembayaran bulanan per metode pembayaran

SQL Part 2 (R7):
```sql
SELECT
    date_trunc('month', so.created_at AT TIME ZONE 'UTC')::date AS order_month,
    p.payment_method,
    COUNT(*) AS total_payments,
    COUNT(*) FILTER (WHERE p.payment_status = 'PAID') AS paid,
    COUNT(*) FILTER (WHERE p.payment_status = 'REFUNDED') AS refunded,
    COUNT(*) FILTER (WHERE p.payment_status = 'FAILED') AS failed,
    COUNT(*) FILTER (WHERE p.payment_status = 'PENDING') AS pending,
    ROUND(100.0 * COUNT(*) FILTER (WHERE p.payment_status = 'PAID')
          / NULLIF(COUNT(*) FILTER (WHERE p.payment_status <> 'PENDING'), 0), 2) AS paid_pct,
    COALESCE(SUM(p.amount) FILTER (WHERE p.payment_status = 'PAID'), 0) AS net_revenue,
    ROUND(AVG(EXTRACT(EPOCH FROM p.paid_at - so.created_at) / 60)::NUMERIC, 1) AS avg_settle_minutes
FROM payments p
JOIN service_orders so ON so.order_id = p.order_id
GROUP BY order_month, p.payment_method
ORDER BY order_month, net_revenue DESC;
```

Butuh hitungan per status pembayaran dan `SUM` per bulan dan metode pembayaran.

In [24]:
sc, ctx = start('PF-10-Q02')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "payment": {"$exists": true}
  },
  "fields": ["created_at", "payment"]
}
'''))
result = timed(lambda: grab.monthly_payment_summary(**ctx['params']), 'view analytics/payments_by_month_method')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/payments_by_month_method: 113.0 ms


,order_month,payment_method,total_payments,paid,refunded,failed,pending,paid_pct,net_revenue,avg_settle_minutes
0,2026-05-01,E_WALLET,148263,125607,18237,4419,0,84.72,5.902812e+09,0.5
1,2026-05-01,CASH,99650,88169,0,11481,0,88.48,4.135309e+09,33.9
2,2026-05-01,PAYLATER,49245,41693,6062,1490,0,84.66,1.959677e+09,0.5
3,2026-05-01,CARD,33059,28002,4021,1036,0,84.70,1.318878e+09,0.5
4,2026-06-01,E_WALLET,143706,121622,17843,4241,0,84.63,5.713362e+09,0.5



PF-10-Q02: lolos validasi = True.


### PF-10-Q03 (R8): 10 outlet dengan pendapatan makanan terbesar

SQL Part 2 (R8):
```sql
SELECT
    mo.merchant_outlet_id,
    mo.outlet_name,
    mo.outlet_status,
    sa.city,
    sa.area_name,
    COUNT(*) AS completed_orders,
    COALESCE(SUM(p.amount) FILTER (WHERE p.payment_status = 'PAID'), 0) AS net_revenue,
    ROUND(AVG(p.amount), 2) AS avg_order_value,
    ROUND(100.0 * COUNT(*) FILTER (WHERE p.payment_status = 'REFUNDED') / COUNT(*), 2) AS refund_rate_pct
FROM merchant_outlets mo
JOIN locations l ON l.location_id = mo.location_id
JOIN service_areas sa ON sa.service_area_id = l.service_area_id
JOIN food_orders f ON f.merchant_outlet_id = mo.merchant_outlet_id
JOIN service_orders so ON so.order_id = f.order_id
JOIN payments p ON p.order_id = f.order_id
WHERE so.order_status = 'COMPLETED'
GROUP BY mo.merchant_outlet_id, mo.outlet_name, mo.outlet_status, sa.city, sa.area_name
ORDER BY net_revenue DESC, mo.merchant_outlet_id
LIMIT 10;
```

Butuh `SUM` per outlet, diambil 10 teratas, lalu data outlet dan area diambil dari `grab_geo`.

In [25]:
sc, ctx = start('PF-10-Q03')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "service_type": "FOOD",
    "order_status": "COMPLETED"
  },
  "fields": ["food.merchant_outlet_id", "total_amount", "payment"]
}
'''))
result = timed(lambda: grab.top_outlets_by_food_revenue(**ctx['params']), 'view analytics/food_revenue_by_outlet')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view analytics/food_revenue_by_outlet: 3,630.2 ms


,merchant_outlet_id,outlet_name,outlet_status,city,area_name,completed_orders,net_revenue,avg_order_value,refund_rate_pct
0,OUT0006321,Seblak Mitra,OPEN,Jakarta,Kelapa Gading,23,2158000.0,98065.22,4.35
1,OUT0042543,Ayam Geprek Jabal,OPEN,Jakarta,Pasar Minggu,23,2130500.0,92630.43,0.00
2,OUT0032324,Pempek Pak Bakiman,OPEN,Jakarta,Kelapa Gading,20,2108500.0,105425.00,0.00
3,OUT0029987,Mie Ayam Puji,OPEN,Jakarta,Cilandak,24,2105500.0,87729.17,0.00
4,OUT0030640,Sate Bu Rini,OPEN,Surabaya,Wonokromo,28,2094000.0,77214.29,3.57



PF-10-Q03: lolos validasi = True.


### PF-09-Q02 (R9): Pasokan outlet dibandingkan permintaan makanan per wilayah, Oktober 2026

SQL Part 2 (R9):
```sql
WITH area_outlets AS (
    SELECT
        l.service_area_id,
        COUNT(*) AS total_outlets,
        COUNT(*) FILTER (WHERE mo.outlet_status = 'OPEN') AS open_outlets
    FROM merchant_outlets mo
    JOIN locations l ON l.location_id = mo.location_id
    GROUP BY l.service_area_id
),
area_orders AS (
    SELECT
        l.service_area_id,
        COUNT(*) AS food_orders
    FROM food_orders f
    JOIN service_orders so ON so.order_id = f.order_id
    JOIN merchant_outlets mo ON mo.merchant_outlet_id = f.merchant_outlet_id
    JOIN locations l ON l.location_id = mo.location_id
    WHERE so.created_at >= '2026-10-01 00:00:00+00'
      AND so.created_at < '2026-11-01 00:00:00+00'
    GROUP BY l.service_area_id
)
SELECT
    sa.city,
    sa.area_name,
    sa.area_status,
    COALESCE(ao.total_outlets, 0) AS total_outlets,
    COALESCE(ao.open_outlets, 0) AS open_outlets,
    COALESCE(od.food_orders, 0) AS food_orders,
    ROUND(od.food_orders::NUMERIC / NULLIF(ao.open_outlets, 0), 2) AS orders_per_open_outlet
FROM service_areas sa
LEFT JOIN area_outlets ao ON ao.service_area_id = sa.service_area_id
LEFT JOIN area_orders od ON od.service_area_id = sa.service_area_id
ORDER BY orders_per_open_outlet DESC NULLS LAST, sa.city, sa.area_name;
```

Butuh dua hitungan dari dua database, yaitu outlet per area dari `grab_geo` dan food order per area dari `grab_orders`, yang lalu digabung seperti `LEFT JOIN`.

In [26]:
sc, ctx = start('PF-09-Q02')
explain(ORDERS, json.loads('''
{
  "selector": {
    "type": "service_order",
    "service_type": "FOOD",
    "created_at": {"$gte": "2026-10-01", "$lt": "2026-11-01"}
  },
  "fields": ["food.merchant_service_area_id"]
}
'''))
result = timed(lambda: grab.outlet_supply_vs_food_demand(**ctx['params']), 'view geo/outlets_by_area_status + analytics/food_orders_by_day_area')
display(pd.DataFrame(result).head())
finish(sc, ctx, result)

POST /grab_orders/_explain
-> tanpa index Mango (_all_docs): CouchDB akan memeriksa setiap dokumen di database
view geo/outlets_by_area_status + analytics/food_orders_by_day_area: 366.3 ms


,city,area_name,area_status,total_outlets,open_outlets,food_orders,orders_per_open_outlet
0,Jakarta,Menteng,ACTIVE,2635,2118,5507,2.60
1,Bandung,Lengkong,ACTIVE,2638,2083,5340,2.56
2,Jakarta,Pasar Minggu,ACTIVE,2581,2028,5190,2.56
3,Bandung,Buahbatu,ACTIVE,2665,2134,5379,2.52
4,Bandung,Sukajadi,ACTIVE,2669,2141,5388,2.52



PF-09-Q02: lolos validasi = True.


## 6. Ringkasan

In [27]:
WAY = {'PF-01-Q01': 'Mango', 'PF-05-Q01': 'Mango',
       'PF-01-Q02': 'view (sort Mango butuh index)', 'PF-03-Q01': 'view (Mango tanpa index)',
       'PF-06-Q02': 'Mango + view + _bulk_docs', 'PF-07-Q01': 'Mango + view + DELETE',
       'PF-07-Q02': 'Mango + view (DELETE ditolak)', 'PF-06-Q01': 'Mango + _bulk_docs'}
summary = pd.DataFrame([{
    'operasi': op.operation_id, 'Part 2': op.part2_query or '', 'deskripsi': op.description,
    'cara': WAY.get(op.operation_id, 'view (agregasi)' if op.kind == 'analytics' else 'Mango + PUT'),
    'lolos validasi': RESULTS.get(op.operation_id)} for op in OPERATIONS])
display(summary)
print(f"{sum(bool(v) for v in RESULTS.values())}/{len(OPERATIONS)} operasi lolos validasi")

,operasi,Part 2,deskripsi,cara,lolos validasi
0,PF-01-Q01,,"Read one order with its subtype, payment, trip...",Mango,True
1,PF-01-Q02,,Read a customer's 20 latest orders,view (sort Mango butuh index),True
2,PF-01-Q03,W1,Create a confirmed order with its payment and ...,Mango + PUT,True
3,PF-02-Q01,W2,Offer a trip to an available driver,Mango + PUT,True
4,PF-02-Q02,W3,Driver accepts the offer and starts the trip,Mango + PUT,True
5,PF-02-Q03,W4,Driver rejects the offer,Mango + PUT,True
6,PF-02-Q04,R4,Trips offered more than twice and eventually a...,view (agregasi),True
7,PF-03-Q01,,Read the status of a trip by trip_id,view (Mango tanpa index),True
8,PF-03-Q02,W5,Complete a trip and make the driver available,Mango + PUT,True
9,PF-04-Q01,W7,Non-cash checkout that fails once and succeeds...,Mango + PUT,True


25/25 operasi lolos validasi


Beberapa hal tentang Mango yang kelihatan dari notebook ini:
- Kalau tidak ada index, batasi `_id` pakai rentang (`$gt`/`$lt`). Cari area Tebet jadi cuma cek 23 dokumen,
  bukan 460 ribu.
- Index baru dipakai kalau semua field index-nya ada di selector. Field yang tidak difilter bisa ditambahkan
  dengan `$exists: true`.
- `$in` pada `_id` tidak pakai index, jadi lebih baik pakai `_id` tunggal atau rentang.
- `sort` harus ada index-nya, kalau tidak CouchDB langsung menolak.
- `$elemMatch` dipakai untuk mencari di dalam array (trip, assignment, kendaraan). Kalau digabung dengan
  `_id`, cuma satu dokumen yang dicek.
- Mango tidak bisa agregasi, join, atau menulis data. Untuk itu tetap pakai view dan `PUT`, `_bulk_docs`,
  atau `DELETE`.
- Biasakan cek `_explain` atau `execution_stats` untuk memastikan index yang dipakai sudah benar.